<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/layer23_kauldron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# HuggingFace authentication
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# Layer 23 TitansBlock Training on Precomputed Activations

Пайплайн обучения обучаемого блока **TitansBlock** на датасете готовых активаций
из стандартных слоёв Gemma 1B перед выбранным слоем.

**Архитектура:**
- **Trainable:** TitansBlock слои (определяются `titans_first_layer`)
- **Frozen:** оставшиеся слои Gemma после последнего TitansBlock + `final_norm` + `embedder`

**Датасет:** `veriga/openwebtext-gemma3-tokenized-1024-activations-layer23`
- Активации precomputed Gemma слоёв 0–22, хранятся в parquet
- Ключи батча: `activations`, `tokens`, `mask`

In [ ]:
# 0. Environment Setup
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog
!pip install -q flax optax seqio
!pip install importlib_resources

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git

## Start

In [ ]:
import os
os._exit(0)

In [ ]:
import sys
import os
sys.path.append(os.getcwd())

import jax
import jax.numpy as jnp
import optax
import dataclasses
import numpy as np
import os
import orbax.checkpoint as ocp
import shutil

from gemma import gm

# Our custom Titans integration
import importlib

%cd /content/Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
import titans_tree_utils
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    load_all_phase1_layers,
    reconstruct_opt_params, schedule,
)

# Layer 23 model and dataset
import layer23_kauldron
importlib.reload(layer23_kauldron)
from layer23_kauldron import (
    Layer23Model, Layer23InitTransform,
    make_layer23_optimizer, get_activation_dataset,
)

# ═══════════════════════════════════════════════════════════
# FIRST_RUN = True  → загрузить Phase 1 чекпойнт из HF
# FIRST_RUN = False → загрузить Phase 2 чекпойнт из HF
# ═══════════════════════════════════════════════════════════
FIRST_RUN = True

HF_CKPT_REPO = "veriga/titans-checkpoints"

# JAX memory settings
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true "
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728"
)
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

## 2. Гиперпараметры

In [ ]:
batch_size = 8
max_length = 1024
total_steps = 60000
embed_dim = 1152

# titans_first_layer: менять для разного количества обучаемых TitansBlock
# 23 → layer 23 only  (1 TitansBlock + 2 Gemma blocks)
# 17 → layers 17,23   (2 TitansBlock + 6 Gemma blocks)
# 11 → layers 11,17,23 (3 TitansBlock + 10 Gemma blocks)
titans_first_layer = 23

_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_first_layer)
print(f"Active Titans layers: {active_titans_layers}")

In [ ]:
experimental_config = {
    # ═══ Архитектура ═══
    'heads': 8,
    'dim_head': 128,
    'chunk_size': 32,
    'mlp_depth': 2,

    # ═══ Training специфика ═══
    'max_grad_norm': 0.5,
    'elastic_net_lambda': 0.005,
    'huber_loss_delta': 0.1,
    'diff_view': False,
    'is_look_ahead': False,
    'adaptive_max_lr': 5e-4,
}

WARMUP = 500

b1_schedule = optax.linear_schedule(
    init_value=0.7,
    end_value=0.90,
    transition_steps=2000,
    transition_begin=WARMUP
)

opt_params = {
    "lr_muon": optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=1e-5,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=5e-6
    ),
    "beta": 0.90,
    "lr_adam": optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=5e-5,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=5e-6,
    ),
    "adam_b1": b1_schedule,
    "adam_b2": 0.85,
    "lr_gate": optax.warmup_cosine_decay_schedule(
        init_value=5e-4,
        peak_value=5e-4,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=5e-4,
    ),
    "gate_b1": b1_schedule,
    "gate_b2": 0.95,
    "every_k_schedule": 4
}

neural_mem_kwargs = {**experimental_config, 'every_k_schedule': opt_params['every_k_schedule']}

from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config

titans_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    training_phase=2,
    titans_layer_indices=active_titans_layers,
    titans_first_layer=titans_first_layer,
    neural_mem_kwargs=neural_mem_kwargs,
)

model = Layer23Model(
    config=titans_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
)

print(f"Model created: Layer23Model (subclass of Gemma3_1B_Titans)")
print(f"  titans_first_layer: {titans_first_layer}")
print(f"  active_titans_layers: {active_titans_layers}")
print(f"  training_phase: 2")
print(f"  neural_mem_kwargs keys: {list(neural_mem_kwargs.keys())}")

## 3. Загрузка весов

In [ ]:
import hf_checkpoint
importlib.reload(hf_checkpoint)
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    reconstruct_opt_params, schedule,
    load_all_phase1_layers,
)

In [ ]:
def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

merged_params = None
workdir_name = 'titans_workdir_layer23'
workdir = os.path.abspath(f'./{workdir_name}')
workdir_checkpoints = os.path.join(workdir, "checkpoints")
init_transform = None

if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"\U0001F4C1 Найдена директория {workdir_checkpoints}. Пропускаем загрузку весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
else:
    # Определяем, какую фазу загружать
    load_phase = 1 if FIRST_RUN else 2
    phase_label = f"Phase {load_phase}"

    # ── Авто-определение последнего чекпойнта ──
    last_meta = load_last_metadata(
        repo_id=HF_CKPT_REPO,
        phase=load_phase,
        token=userdata.get('HF_TOKEN'),
    )

    if last_meta is not None:
        # Восстанавливаем experimental_config
        experimental_config = last_meta.get("experimental_config", experimental_config)
        print(f"\U0001F4CB Restored experimental_config: {experimental_config}")

        # Восстанавливаем opt_params: schedules → callable
        if "opt_params" in last_meta:
            opt_params = reconstruct_opt_params(last_meta["opt_params"])
            print(f"\U0001F4CB Restored opt_params with schedules: {list(opt_params.keys())}")

        # Восстанавливаем warm_up
        if "warm_up" in last_meta:
            WARMUP = last_meta["warm_up"]
            print(f"\U0001F4CB Restored warm_up: {WARMUP}")
    else:
        print("\u26A0\uFE0F Last metadata not found — using current notebook values")

    if FIRST_RUN:
        # ── Загружаем веса для активных Titans слоёв из Phase 1 ──
        loaded_titans_params = load_all_phase1_layers(
            repo_id=HF_CKPT_REPO,
            titans_first_layer=titans_first_layer,
            local_dir=".",
            token=userdata.get('HF_TOKEN'),
        )

        if loaded_titans_params is not None:
            active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
            loaded_titans_params = {
                k: v for k, v in loaded_titans_params.items()
                if k in active_layer_keys
            }
            print(f"Merging Titans weights for: {sorted(loaded_titans_params.keys())}")

            print("Loading Gemma base weights...")
            original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

            merged_params = titans_tree_utils.merge_titans_params(
                original_params, loaded_titans_params, remove_dead_attn=True
            )
            print(f"\u2705 Phase 1 weights loaded from HF and merged.")
        else:
            print("\u26A0\uFE0F Не найдено обученных слоёв Phase 1 на HF!")
    else:
        # ── Загружаем комбинированный чекпойнт Phase 2 ──
        if last_meta is not None:
            titans_first_layer = last_meta.get("first_layer", titans_first_layer)
            total_steps = last_meta.get("total_steps", total_steps)
            print(f"\U0001F4CB Checkpoint: {last_meta.get('checkpoint')}")

        ckpt_dir = load_checkpoint_from_hf(
            repo_id=HF_CKPT_REPO,
            phase=2,
            first_layer=titans_first_layer,
            total_steps=total_steps,
            local_dir=".",
        )

        if ckpt_dir is not None:
            print("Loading Gemma base weights...")
            original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

            print(f"Loading Phase 2 Titans weights from HF...")
            loaded_titans_params = load_titans_weights(ckpt_dir)

            active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
            loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}
            print(f"Merging Titans weights for: {sorted(active_layer_keys)}")

            merged_params = titans_tree_utils.merge_titans_params(
                original_params, loaded_titans_params, remove_dead_attn=True
            )
            print(f"\u2705 Phase 2 weights loaded from HF and merged.")
        else:
            print(f"\u26A0\uFE0F Чекпойнт Phase 2 не найден на HF!")

    # Создаём init_transform из merged_params
    if merged_params is not None:
        init_transform = Layer23InitTransform(merged_params, titans_first_layer=titans_first_layer)
        print(f"\u2705 Layer23InitTransform created (first_layer={titans_first_layer})")

## 4. Датасет (Activations from Parquet)

In [ ]:
# Датасет предвычисленных активаций слоя 23
# Репозиторий: veriga/openwebtext-gemma3-tokenized-1024-activations-layer23
# Подкаталог: data
# Ключи: activations (float32 [1024, 1152]), tokens (int32 [1024]), mask (int32 [1024])

print(f"Dataset: veriga/openwebtext-gemma3-tokenized-1024-activations-layer23/data")
print(f"Batch size: {batch_size}, Max seq len: {max_length}")

## 5. Оптимизатор (замороженные слои + trainable TitansBlock)

In [ ]:
from routing_optimizer import make_routing_optimizer

huber_loss_delta = experimental_config['huber_loss_delta']

# Внутренний Titans optimizer (M3 + Adam для параметров TitansBlock)
titans_optimizer = make_routing_optimizer(opt_params)

# Обёртка: только TitansBlock слои обучаются, остальные заморожены
optimizer = make_layer23_optimizer(titans_optimizer, titans_first_layer=titans_first_layer)

print(f"Optimizer created (first_layer={titans_first_layer})")
print(f"  - trainable: {active_titans_layers}")
print(f"  - frozen: all other layers + final_norm + embedder")

## 6. Loss и метрики

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext, kd

# LM loss — основной сигнал
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# ── Метрики ──

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class AdamLearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(opt_params['lr_adam'](step)))

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            return cls()
        def merge(self, other):
            return self

    def get_state(self, **kwargs) -> State:
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LMAccuracy(kd.metrics.Metric):
    acc: kd.kontext.Key = "preds.layer_losses['lm_accuracy']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        pass
    def get_state(self,*, acc) -> State:
        return self.State.from_values(values=acc)

train_metrics = {
    "LM/accuracy": LMAccuracy(),
    "LM/adam_lr": AdamLearningRateMetric(),
    "tpu_memory": TPUMemoryMetric()
}

train_summaries = {}
for layer in active_titans_layers:
    key = f"Gates_Dist_{layer}"
    train_summaries[key] = kd.summaries.HistogramSummary(tensor=f"params.layer_{layer}.memory_gate_proj.kernel")

print(f"Losses, metrics, summaries configured. Summaries for: {active_titans_layers}")

## 7. Trainer

In [ ]:
# Проверка перед обучением
if merged_params is not None:
    for _layer in active_titans_layers:
        _gk = merged_params[f'layer_{_layer}']['memory_gate_proj']['kernel']
        print(f"Layer {_layer} gate — Mean: {_gk.mean():.4f}, Std: {_gk.std():.4f}, Min: {_gk.min():.4f}, Max: {_gk.max():.4f}")
else:
    print("merged_params is None — checkpoint will be loaded by Kauldron from workdir")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_ds = get_activation_dataset(
    repo_id="veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    data_dir="data",
    batch_size=batch_size,
    max_seq_len=max_length,
    embed_dim=embed_dim,
)

trainer = kd.train.Trainer(
    seed=42,
    workdir=workdir,
    train_ds=train_ds,
    model=model,
    init_transform=init_transform,
    num_train_steps=total_steps,
    train_losses=train_losses,
    train_metrics=train_metrics,
    train_summaries=train_summaries,
    optimizer=optimizer,
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),
)

print(f"Trainer initialized. workdir: {workdir_name}")

In [ ]:
trainer.num_train_steps

## 8. TensorBoard

Если у вас уже запущено длительное обучение trainer.train() в ячейке, и нужно запустить заново Tensorboard, вам нужно воспользоваться Терминалом Colab:

```
fuser -k 6006/tcp && tensorboard --logdir /content/Titans_jax/titans_workdir_layer23/ --port 6006 &
```

In [ ]:
%reload_ext tensorboard
from tensorboard import notebook
notebook.list()

In [ ]:
!rm -rf /tmp/.tensorboard-info/*
!fuser -k 6006/tcp

In [ ]:
%tensorboard --logdir ./{workdir_name}/ --port=6006

## 9. Обучение

In [ ]:
state, aux = trainer.train()

## TPU Memory & Continue Training

In [ ]:
import jax

def print_tpu_mem_tpu_native():
    d = jax.devices()[0]
    ms = d.memory_stats() or {}
    used = ms.get("bytes_in_use", 0)
    peak = ms.get("peak_bytes_in_use", 0)
    limit = ms.get("bytes_limit", 0)
    reserved = ms.get("bytes_reserved", 0)
    reservable_limit = ms.get("bytes_reservable_limit", 0)
    largest_free = ms.get("largest_free_block_bytes", 0)
    print(f"Device: {d}")
    print(f"used_gb:               {used / 1e9:.2f}")
    print(f"peak_used_gb:          {peak / 1e9:.2f}")
    print(f"bytes_limit_gb:        {limit / 1e9:.2f}")
    print(f"bytes_reserved_gb:     {reserved / 1e9:.2f}")
    print(f"reservable_limit_gb:   {reservable_limit / 1e9:.2f}")
    print(f"largest_free_block_gb: {largest_free / 1e9:.2f}")
    if reservable_limit > 0:
        reservable_free = max(reservable_limit - reserved, 0)
        print(f"reservable_free_gb:    {reservable_free / 1e9:.2f}")

print_tpu_mem_tpu_native()

In [ ]:
import jax
import gc

try:
    del state
    del aux
except NameError:
    pass

for device in jax.devices():
    if hasattr(device, 'live_arrays'): device.live_arrays().clear()
    if hasattr(device, 'live_buffers'): device.live_buffers().clear()
    if hasattr(device, 'default_memory_tracker'): device.default_memory_tracker().clear()

jax.clear_caches()
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")
print_tpu_mem_tpu_native()

In [ ]:
trainer = trainer.replace(num_train_steps=50000)
state, aux = trainer.train()

## 10. Сохранение весов

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # Извлекаем только обучаемые TitansBlock слои из params
    params = state.params
    titans_params = {f'layer_{l}': params[f'layer_{l}'] for l in active_titans_layers}
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved Titans weights to {save_path}")

new_weights_name = f"saved_titans_layer{titans_first_layer}_{total_steps}"
save_titans_weights(state, f"./{new_weights_name}")

# Upload to HuggingFace Hub
from google.colab import userdata

# 1. Загружаем чекпойнт (веса + метадата)
save_checkpoint_to_hf(
    save_dir=f"./{new_weights_name}",
    repo_id=HF_CKPT_REPO,
    phase=2,
    warm_up=WARMUP,
    experimental_config=experimental_config,
    opt_params=opt_params,
    first_layer=titans_first_layer,
    total_steps=total_steps,
    token=userdata.get('HF_TOKEN'),
)

# 2. Обновляем «указатель» на последний эксперимент
save_last_metadata(
    repo_id=HF_CKPT_REPO,
    phase=2,
    warm_up=WARMUP,
    first_layer=titans_first_layer,
    total_steps=total_steps,
    experimental_config=experimental_config,
    opt_params=opt_params,
    token=userdata.get('HF_TOKEN'),
)
print(f"\u2705 Layer {titans_first_layer} checkpoint + metadata uploaded to {HF_CKPT_REPO}")

## 11. Training Report

Сохраняем результаты обучения на HF, привязанные к гиперпараметрам:
- JSON-отчёт с loss-статистикой и гиперпараметрами
- PNG-график loss

In [ ]:
from hf_checkpoint import save_training_report, read_tensorboard_losses

# ── Читаем loss из TensorBoard ──
loss_history = read_tensorboard_losses(
    workdir=os.path.abspath(f'./{workdir_name}'),
    tag="lm_loss",
)
print(f"\U0001F4CA Прочитано {len(loss_history)} точек loss из TensorBoard")

if loss_history:
    vals = [e["value"] for e in loss_history]
    print(f"   Loss: {vals[0]:.4f} → {vals[-1]:.4f}")
    print(f"   Last 500 avg: {np.mean(vals[-500:]):.4f}" if len(vals) >= 500 else "")

# ── Загружаем отчёт на HF ──
save_training_report(
    repo_id=HF_CKPT_REPO,
    phase=2,
    first_layer=titans_first_layer,
    total_steps=total_steps,
    loss_history=loss_history,
    extra_metrics={
        "batch_size": batch_size,
        "max_length": max_length,
        "active_titans_layers": list(active_titans_layers),
        "model": "Layer23Model",
        "dataset": "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    },
    experimental_config=experimental_config,
    opt_params=opt_params,
    token=userdata.get('HF_TOKEN'),
)
print("\u2705 Training report uploaded!")